In [1]:
# =========================
# INSTALLS (if needed)
# =========================
# !pip install requests beautifulsoup4 pandas


# =========================
# IMPORTS
# =========================
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time
import random
import json
import os


# =========================
# CONFIG
# =========================
CDX_URL = "http://web.archive.org/cdx/search/cdx"
CHECKPOINT_FILE = "scraper_checkpoint.json"

TIME_REGEX = re.compile(r"\d{1,2}:\d{2}\s?(?:AM|PM)", re.IGNORECASE)


# =========================
# WAYBACK SNAPSHOTS
# =========================
def get_snapshots():
    params = {
        "url": "riverviewtheater.com/",
        "from": "2021",
        "to": "2025",
        "output": "json",
        "collapse": "digest"
    }

    res = requests.get(CDX_URL, params=params, timeout=30)
    data = res.json()

    return data[1:]


def reduce_snapshots(snapshots):
    seen = set()
    filtered = []

    for row in snapshots:
        timestamp = row[1]
        date = timestamp[:8]

        week_key = date[:6] + str(int(date[6:8]) // 7)

        if week_key not in seen:
            seen.add(week_key)
            filtered.append(row)

    return filtered


# =========================
# URL BUILDER
# =========================
def build_url(timestamp):
    return f"https://web.archive.org/web/{timestamp}/https://riverviewtheater.com/"


# =========================
# ROBUST FETCHER
# =========================
session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (compatible; RiverviewScraper/1.0)"
})

def safe_get(url, retries=5):
    for i in range(retries):
        try:
            res = session.get(url, timeout=30)

            if res.status_code == 200:
                return res

            print(f"HTTP {res.status_code} retry {i+1}")

        except requests.exceptions.RequestException as e:
            print(f"Network error retry {i+1}: {e}")

        time.sleep((2 ** i) + random.uniform(0.5, 2.0))

    return None


# =========================
# PARSER (STRUCTURE-AWARE)
# =========================
def clean_text(x):
    return re.sub(r"\s+", " ", x).strip()


def is_noise(text):
    t = text.upper()

    bad = [
        "NOW PLAYING",
        "UPCOMING",
        "RIVERVIEW",
        "TICKETS",
        "SHOWTIMES",
        "SATURDAY", "SUNDAY", "MONDAY",
        "TUESDAY", "WEDNESDAY", "THURSDAY", "FRIDAY"
    ]

    return any(b in t for b in bad)


def parse_riverview(html):
    soup = BeautifulSoup(html, "html.parser")
    results = []

    blocks = soup.find_all(["div", "section", "li", "td", "tr"])

    for block in blocks:
        text = clean_text(block.get_text(" "))

        if not text or len(text) < 5:
            continue

        if is_noise(text):
            continue

        times = TIME_REGEX.findall(text)
        if not times:
            continue

        title = TIME_REGEX.sub("", text)
        title = clean_text(title)

        if len(title) < 3 or len(title) > 80:
            continue

        if is_noise(title):
            continue

        for t in times:
            results.append({
                "film_title": title,
                "showtime": t
            })

    return results


# =========================
# CHECKPOINT SYSTEM
# =========================
def load_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, "r") as f:
            return json.load(f).get("last_index", 0)
    return 0


def save_checkpoint(i):
    with open(CHECKPOINT_FILE, "w") as f:
        json.dump({"last_index": i}, f)


# =========================
# MAIN SCRAPING PIPELINE
# =========================
snapshots = get_snapshots()
snapshots = reduce_snapshots(snapshots)

print("Snapshots found:", len(snapshots))
print("Reduced snapshots:", len(snapshots))


rows = []
start_index = load_checkpoint()
print(f"Resuming from: {start_index}")


for i, row in enumerate(snapshots[start_index:], start=start_index):

    timestamp = row[1]
    url = build_url(timestamp)

    res = safe_get(url)

    if not res:
        print(f"[{i}] FAILED {timestamp}")
        save_checkpoint(i)
        continue

    try:
        parsed = parse_riverview(res.text)

        for item in parsed:
            rows.append({
                "theater_name": "Riverview Theater",
                "snapshot_date": timestamp[:8],
                "film_title": item["film_title"],
                "screening_time": item["showtime"],
                "source_url": url
            })

        print(f"[{i}] OK {timestamp} -> {len(parsed)} showings")

    except Exception as e:
        print(f"[{i}] PARSE ERROR {timestamp}: {e}")

    save_checkpoint(i)

    time.sleep(random.uniform(1.5, 3.5))


# =========================
# OUTPUT DATAFRAME
# =========================
df = pd.DataFrame(rows)
df

Snapshots found: 121
Reduced snapshots: 121
Resuming from: 0
[0] OK 20210104072028 -> 0 showings
[1] OK 20210123004704 -> 0 showings
[2] OK 20210204194200 -> 0 showings
[3] OK 20210211232820 -> 0 showings
[4] OK 20210218160954 -> 0 showings
[5] OK 20210305002439 -> 0 showings
[6] OK 20210314183542 -> 0 showings
[7] OK 20210329124607 -> 0 showings
[8] OK 20210417043157 -> 0 showings
[9] OK 20210514154353 -> 0 showings
[10] OK 20210524210213 -> 0 showings
[11] OK 20210619204836 -> 2 showings
[12] OK 20210630192645 -> 1 showings
[13] OK 20210706053925 -> 0 showings
[14] OK 20210802080659 -> 1 showings
[15] OK 20210918125938 -> 8 showings
[16] OK 20211011130655 -> 1 showings
[17] OK 20211023225923 -> 10 showings
[18] OK 20211121150950 -> 6 showings
[19] OK 20211222225012 -> 6 showings
[20] OK 20220129094004 -> 5 showings
[21] OK 20220209062107 -> 3 showings
[22] OK 20220218065900 -> 8 showings
[23] OK 20220313123011 -> 6 showings
[24] OK 20220327040530 -> 5 showings
[25] OK 20220406212142 

,theater_name,snapshot_date,film_title,screening_time,source_url
0,Riverview Theater,20210619,In The Heights,3:30PM,https://web.archive.org/web/20210619204836/htt...
1,Riverview Theater,20210619,In The Heights,6:40PM,https://web.archive.org/web/20210619204836/htt...
2,Riverview Theater,20210630,The Croods: A New Age,12:30PM,https://web.archive.org/web/20210630192645/htt...
3,Riverview Theater,20210802,Jungle Cruise,6:30PM,https://web.archive.org/web/20210802080659/htt...
4,Riverview Theater,20210918,", , ,",1:00PM,https://web.archive.org/web/20210918125938/htt...
...,...,...,...,...,...
590,Riverview Theater,20251207,Wicked: For Good,3:30PM,https://web.archive.org/web/20251207064740/htt...
591,Riverview Theater,20251217,A Christmas Story (1983),12:30PM,https://web.archive.org/web/20251217091323/htt...
592,Riverview Theater,20251217,It's A Wonderful Life (1946),2:30PM,https://web.archive.org/web/20251217091323/htt...
593,Riverview Theater,20251217,Elf (2003),5:00PM,https://web.archive.org/web/20251217091323/htt...


In [ ]:
df.to_csv('Riverview.csv', index=False)